# LifeMate (code baseline): bước 2, LLM

Train LoRA cho **Qwen3-4B-Instruct-2507** trên quỹ đạo gold của `splits/train2_ids.txt`, chấm trên `splits/dev2_ids.txt` (300 mẫu, khuôn câu không có trong train2), rồi tạo bài nộp public bằng ASR đã fine-tune ở `colab_baseline.ipynb`.

**Lần chạy `asraug`**: thêm khoảng 205 quỹ đạo dùng lời chép ASR thật (những mẫu train2 mà ASR chưa từng nghe), cùng đáp án, để model quen lỗi ASR. Lần trước (không có phần này): dev2 oracle 59,7, dev2 ASR 47,1, public 47.

Chuẩn bị:
1. **Runtime → GPU A100.**
2. Trên Drive đã có `MyDrive/zalo_lifemate/train+public_test.zip`, `models/phowhisper_large_ft_fp16/`, `asr/asr_dev_noprompt.json` và `asr/asr_cache_public_test_noprompt.json` (từ các lần chạy trước), secret `GH_TOKEN` như trước.

Bấm **Chạy tất cả**. Dự kiến khoảng 2 giờ: cài đặt 10 phút, train khoảng 70 phút, chấm dev2 10 phút, chạy public 15 phút.
Mọi kết quả mang tên `RUN` nên không đè lần trước; chạy lại sẽ bỏ qua bước nào đã có (ASR public, adapter LoRA).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Code + data + thư viện

In [ ]:
import os, json
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
RUN = "asraug"   # tên lần chạy: adapter, kết quả dev2, bài nộp đều theo tên này
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
os.environ["PYTHONPATH"] = f"{REPO}/src"
ASR_DRIVE = f"{DRIVE}/models/phowhisper_large_ft_fp16"
ASR_DEV = f"{DRIVE}/asr/asr_dev_noprompt.json"   # lời chép của các mẫu mà ASR chưa từng train
assert os.path.isfile(f"{DRIVE}/train+public_test.zip"), f"chưa có {DRIVE}/train+public_test.zip trên Drive"
assert os.path.isdir(ASR_DRIVE), f"chưa có {ASR_DRIVE}: chạy colab_baseline.ipynb trước"
assert os.path.isfile(ASR_DEV), f"chưa có {ASR_DEV}: chạy colab_baseline.ipynb trước"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!mkdir -p out/{RUN} && ls train+public_test

In [ ]:
# vllm kéo theo torch của nó; torchao bản của Colab làm peft lỗi import nên gỡ đi
!pip install -q -r requirements.txt vllm trl peft datasets && pip uninstall -y -q torchao
!python -m pytest -q tests 2>&1 | tail -2
# nếu cell sau báo lỗi lạ (SyntaxError ở dòng !, lỗi import): Runtime → Restart session rồi chạy lại từ đầu

## 1. Chép lời public_test, không dùng domain prompt
Trên dev, bỏ prompt cho WER 8,65% so với 8,91% khi có prompt. Lần trước đã làm và lưu lên Drive nên cell này sẽ bỏ qua; nếu chưa có thì mất khoảng 30–35 phút (cứ 50 đoạn in một dòng `[asr] .../1933 ... eta ... min`).

In [ ]:
PUB_ASR = f"{DRIVE}/asr/asr_cache_public_test_noprompt.json"
if os.path.isfile(PUB_ASR):
    print("Đã có trên Drive, bỏ qua:", PUB_ASR)
else:
    # tên file mới: precompute_asr.py dùng tiếp cache cũ nếu --out đã tồn tại
    !python scripts/precompute_asr.py --data ./train+public_test --split public_test --backend hf --model {ASR_DRIVE} --gpus 1 --no-metrics --out out/asr_cache_public_test_noprompt.json 2>&1 | tee out/asr_public.log | grep --line-buffered -E "^\[asr\]|ERROR|Error|Traceback"
    !grep -A 40 Traceback out/asr_public.log | tail -45
    !mkdir -p "{DRIVE}/asr" && cp out/asr_cache_public_test_noprompt.json "{PUB_ASR}" && ls -la "{DRIVE}/asr"

## 2. Dữ liệu SFT + train LoRA
1245 quỹ đạo gold cộng khoảng 205 bản sao dùng lời chép ASR (`--asr-aug`: chỉ mẫu train2, không lấy mẫu dev2). Mỗi quỹ đạo khoảng 5.300 token (dài nhất 8.800). Loss chỉ tính trên lượt assistant (`--assistant-only`). Thanh tiến trình hiện thời gian còn lại; cứ 10 bước in một dòng `{'loss': ...}`.

In [ ]:
!python scripts/build_sft.py --data ./train+public_test --split train --ids-file splits/train2_ids.txt --asr-aug "{ASR_DEV}" --out out/{RUN}/sft_train.jsonl --few-shot-out out/{RUN}/few_shot.jsonl --index-out out/{RUN}/few_shot_index.jsonl

In [ ]:
LORA = f"models/qwen3-4b-{RUN}"      # train_lora.py lưu adapter ở đây và bản đã merge ở LORA + "-merged"
MERGED = LORA + "-merged"
LORA_DRIVE = f"{DRIVE}/lora_{RUN}"
if os.path.isfile(f"{LORA_DRIVE}/adapter_model.safetensors"):
    print("Đã có adapter trên Drive, bỏ qua train:", LORA_DRIVE)
else:
    # 1 GPU x grad-accum 16 = cùng batch hiệu dụng với README (4 GPU x 4)
    !python scripts/train_lora.py --train out/{RUN}/sft_train.jsonl --out {LORA} --epochs 2 --lr 1e-4 --r 16 --max-len 10240 --dtype bfloat16 --grad-accum 16 --assistant-only
    # chỉ adapter (khoảng 130MB) lên Drive; bản merge 8GB dựng lại từ adapter khi cần
    !mkdir -p "{LORA_DRIVE}" && cp {LORA}/adapter_config.json {LORA}/adapter_model.safetensors "{LORA_DRIVE}/" && ls -la "{LORA_DRIVE}"
if not os.path.isdir(MERGED):
    !python scripts/merge_lora.py --adapter "{LORA_DRIVE}" --out {MERGED}
!ls {MERGED}
# config của lần chạy này = configs/baseline.json, chỉ đổi model LLM
CFG = f"out/{RUN}/config.json"
cfg = json.load(open("configs/baseline.json")); cfg["llm_model"] = MERGED
json.dump(cfg, open(CFG, "w"), ensure_ascii=False, indent=1)

## 3. Chấm trên dev2
- **oracle** (300 mẫu, lời thoại chuẩn): đo riêng phần LLM, dùng để so các bản model với nhau.
- **ASR** (64 mẫu `splits/dev2_asr_clean_ids.txt`): chỉ những mẫu mà ASR chưa từng train trên audio của chúng. 236 mẫu còn lại của dev2 có audio nằm trong dữ liệu train ASR nên chép lời gần như hoàn hảo, không phản ánh public.

Dòng `trước:` là lần chạy cũ để so. Chênh dưới khoảng 3 điểm (oracle) hay 6 điểm (ASR, chỉ 64 mẫu) thì coi như ngang nhau.

In [ ]:
!python scripts/run_dev.py --data ./train+public_test --split train --ids-file splits/dev2_ids.txt --oracle --config {CFG} --out-dir out/{RUN}/dev2_oracle 2>&1 | tee out/{RUN}/dev2_oracle.log | grep --line-buffered -E "^\[dev\]|ERROR|Error|Traceback"
!grep -A 40 Traceback out/{RUN}/dev2_oracle.log | tail -45
!python scripts/run_dev.py --data ./train+public_test --split train --ids-file splits/dev2_asr_clean_ids.txt --asr-cache "{ASR_DEV}" --config {CFG} --out-dir out/{RUN}/dev2_asr_clean 2>&1 | tee out/{RUN}/dev2_asr_clean.log | grep --line-buffered -E "^\[dev\]|ERROR|Error|Traceback"
!grep -A 40 Traceback out/{RUN}/dev2_asr_clean.log | tail -45
def line(path):
    s = json.load(open(path))
    return (f"final={s['final']:.2f} n={s['n']} perfect={s['perfect']} zero={s['zero']} "
            f"{ {k: round(v['score'], 1) for k, v in s['by_category'].items()} } p95={s['latency']['p95']:.1f}s")
for name in ("dev2_oracle", "dev2_asr_clean"):
    print(f"{RUN} {name}: {line(f'out/{RUN}/{name}/summary.json')}")
    if os.path.isfile(f"{DRIVE}/runs/{name}/summary.json"):
        print(f"   trước: {line(f'{DRIVE}/runs/{name}/summary.json')}")

In [ ]:
!python scripts/error_buckets.py --run out/{RUN}/dev2_oracle --top 6 --per 2 | head -80
!mkdir -p "{DRIVE}/runs/{RUN}" && cp -r out/{RUN}/dev2_oracle out/{RUN}/dev2_asr_clean "{DRIVE}/runs/{RUN}/" && ls "{DRIVE}/runs/{RUN}"

## 4. Bài nộp public_test

In [ ]:
!python scripts/infer.py --data ./train+public_test --split public_test --config {CFG} --asr-cache "{PUB_ASR}" --out out/{RUN}/public/submission.jsonl --no-resume 2>&1 | tee out/{RUN}/public.log | grep --line-buffered -E "^\[infer\] ([0-9]+/|done)|ERROR|Error|Traceback"
!grep -A 40 Traceback out/{RUN}/public.log | tail -45
!python scripts/check_submission.py --data ./train+public_test --split public_test --sub out/{RUN}/public/submission.jsonl

In [ ]:
!cp out/{RUN}/public/submission.jsonl "{DRIVE}/runs/{RUN}/submission.jsonl" && ls -la "{DRIVE}/runs/{RUN}"
from google.colab import files
files.download(f"out/{RUN}/public/submission.jsonl")

## 5. Gói kết quả để gửi phân tích
Nén dev2 + public + log của lần chạy này thành một file (khung Tệp của Colab không tải được thư mục).

In [ ]:
!cd out/{RUN} && zip -qr ../../runs_{RUN}.zip dev2_oracle dev2_asr_clean public dev2_oracle.log dev2_asr_clean.log public.log && ls -la ../../runs_{RUN}.zip
!cp runs_{RUN}.zip "{DRIVE}/runs/"
from google.colab import files
files.download(f"runs_{RUN}.zip")

## Xong
1. Nộp `submission.jsonl` vừa tải ở mục **Submission** trên portal (tối đa 5 lượt/ngày).
2. Chép `runs_asraug.zip` vào `runs/` trên máy (cũng có bản trên Drive ở `MyDrive/zalo_lifemate/runs/`) rồi báo điểm public.